# 🧠 Agent Components Testing Suite
## Validating Individual Agent Behavior & Contracts

**Notebook Purpose:** Test each agent module individually to ensure they meet Design Truth Contract requirements.

**Scope:**
- ✅ AgentRouter - Task routing
- ✅ ChainOfThoughtAgent - Reasoning chains
- ✅ IterativeAgent - Iterative refinement
- ✅ TaskDecomposer - Task breakdown
- ✅ ParallelTaskExecutor - Concurrent execution
- ✅ ResultSynthesizer - Result combination
- ✅ OutputValidator - Output validation
- ✅ MemoryManager - State management
- ✅ ModelRouter - Model selection

**Version:** 1.0
**Last Updated:** 2025-12-20
**Status:** Component Testing Phase 1

In [ ]:
import sys
import os
import json
import time
import logging
from pathlib import Path
from typing import Dict, List, Any, Tuple
from datetime import datetime
import traceback

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger("AgentComponentsTest")

# Add agents directory to path
agents_path = Path("/workspaces/cognito-stack/agents")
sys.path.insert(0, str(agents_path.parent))

# Test config
TEST_RESULTS = {
    "total_tests": 0,
    "passed": 0,
    "failed": 0,
    "errors": [],
    "start_time": datetime.now(),
    "components_tested": []
}

print("✅ Environment configured")
print(f"   Python: {sys.version}")
print(f"   Agents Path: {agents_path}")
print(f"   Working Directory: {os.getcwd()}")

## 1️⃣ Test Framework & Utilities

In [ ]:
class AgentTestFramework:
    """Base framework for agent component testing"""
    
    def __init__(self):
        self.results = []
        self.metrics = {}
        
    def test_component(self, component_name: str, test_func, *args, **kwargs) -> Dict[str, Any]:
        """Execute a component test and record results"""
        TEST_RESULTS["total_tests"] += 1
        result = {
            "component": component_name,
            "timestamp": datetime.now().isoformat(),
            "status": "pending",
            "duration": 0,
            "error": None
        }
        
        start = time.time()
        try:
            test_func(*args, **kwargs)
            result["status"] = "passed"
            TEST_RESULTS["passed"] += 1
            logger.info(f"✅ {component_name}: PASSED")
        except AssertionError as e:
            result["status"] = "failed"
            result["error"] = str(e)
            TEST_RESULTS["failed"] += 1
            TEST_RESULTS["errors"].append({
                "component": component_name,
                "error": str(e)
            })
            logger.error(f"❌ {component_name}: FAILED - {e}")
        except Exception as e:
            result["status"] = "error"
            result["error"] = str(e)
            TEST_RESULTS["failed"] += 1
            TEST_RESULTS["errors"].append({
                "component": component_name,
                "error": f"Exception: {str(e)}"
            })
            logger.error(f"⚠️  {component_name}: ERROR - {e}")
        finally:
            result["duration"] = time.time() - start
            self.results.append(result)
        
        return result
    
    def assert_contract(self, condition: bool, message: str):
        """Assert a contract condition"""
        assert condition, f"Contract violation: {message}"
    
    def get_summary(self) -> Dict[str, Any]:
        """Get test summary"""
        return {
            "total_tests": TEST_RESULTS["total_tests"],
            "passed": TEST_RESULTS["passed"],
            "failed": TEST_RESULTS["failed"],
            "success_rate": f"{(TEST_RESULTS['passed'] / max(TEST_RESULTS['total_tests'], 1)) * 100:.1f}%",
            "duration": (datetime.now() - TEST_RESULTS["start_time"]).total_seconds(),
            "components_tested": list(set(TEST_RESULTS["components_tested"]))
        }

framework = AgentTestFramework()
print("✅ Test framework initialized")

## 2️⃣ Test AgentRouter Component

**Contract Requirements:**
- Input: Task type (string)
- Output: Agent instance
- Error handling: Return None for unknown types
- Performance: < 100ms routing

In [ ]:
def test_agent_router():
    """Test AgentRouter component"""
    try:
        from agents.agent_router import AgentRouter
        
        # Test 1: Router initialization
        router = AgentRouter()
        framework.assert_contract(
            router is not None,
            "AgentRouter must be instantiable"
        )
        
        # Test 2: Router has route method
        framework.assert_contract(
            hasattr(router, 'route') and callable(router.route),
            "AgentRouter must have route() method"
        )
        
        # Test 3: Route method returns result
        test_task = {
            "type": "analysis",
            "description": "Test task",
            "data": {}
        }
        result = router.route(test_task)
        framework.assert_contract(
            result is not None,
            "Router must return a result"
        )
        
        logger.info(f"   Router result type: {type(result)}")
        TEST_RESULTS["components_tested"].append("AgentRouter")
        
    except ImportError as e:
        logger.warning(f"⚠️  AgentRouter import skipped: {e}")
    except Exception as e:
        logger.error(f"AgentRouter test error: {e}")
        raise

result = framework.test_component("AgentRouter", test_agent_router)
print(f"AgentRouter Test: {result['status'].upper()}")

## 3️⃣ Test TaskDecomposer Component

**Contract Requirements:**
- Input: Complex task description
- Output: List of subtasks with dependencies
- Dependency tracking: Identify task relationships
- Optimization: Minimize task count

In [ ]:
def test_task_decomposer():
    """Test TaskDecomposer component"""
    try:
        from agents.task_decomposer import TaskDecomposer
        
        # Test 1: Decomposer initialization
        decomposer = TaskDecomposer()
        framework.assert_contract(
            decomposer is not None,
            "TaskDecomposer must be instantiable"
        )
        
        # Test 2: Has decompose method
        framework.assert_contract(
            hasattr(decomposer, 'decompose') and callable(decomposer.decompose),
            "TaskDecomposer must have decompose() method"
        )
        
        # Test 3: Decompose complex task
        complex_task = {
            "description": "Analyze customer data, generate report, and send email",
            "priority": "high"
        }
        subtasks = decomposer.decompose(complex_task)
        
        framework.assert_contract(
            isinstance(subtasks, (list, dict)),
            "Decompose must return list or dict of subtasks"
        )
        
        logger.info(f"   Decomposed into {len(subtasks) if isinstance(subtasks, list) else len(subtasks.items())} subtasks")
        TEST_RESULTS["components_tested"].append("TaskDecomposer")
        
    except ImportError as e:
        logger.warning(f"⚠️  TaskDecomposer import skipped: {e}")
    except Exception as e:
        logger.error(f"TaskDecomposer test error: {e}")
        raise

result = framework.test_component("TaskDecomposer", test_task_decomposer)
print(f"TaskDecomposer Test: {result['status'].upper()}")

## 4️⃣ Test ParallelTaskExecutor Component

**Contract Requirements:**
- Input: List of tasks
- Output: List of results in same order
- Concurrency: Execute tasks in parallel
- Error isolation: One failure doesn't stop others

In [ ]:
def test_parallel_executor():
    """Test ParallelTaskExecutor component"""
    try:
        from agents.parallel_task_executor import ParallelTaskExecutor
        
        # Test 1: Executor initialization
        executor = ParallelTaskExecutor()
        framework.assert_contract(
            executor is not None,
            "ParallelTaskExecutor must be instantiable"
        )
        
        # Test 2: Has execute method
        framework.assert_contract(
            hasattr(executor, 'execute') and callable(executor.execute),
            "ParallelTaskExecutor must have execute() method"
        )
        
        # Test 3: Execute multiple tasks
        tasks = [
            {"id": 1, "name": "Task 1", "duration": 0.1},
            {"id": 2, "name": "Task 2", "duration": 0.1},
            {"id": 3, "name": "Task 3", "duration": 0.1}
        ]
        
        start = time.time()
        results = executor.execute(tasks)
        duration = time.time() - start
        
        framework.assert_contract(
            isinstance(results, (list, dict)),
            "Execute must return list or dict of results"
        )
        
        logger.info(f"   Executed {len(tasks)} tasks in {duration:.2f}s")
        TEST_RESULTS["components_tested"].append("ParallelTaskExecutor")
        
    except ImportError as e:
        logger.warning(f"⚠️  ParallelTaskExecutor import skipped: {e}")
    except Exception as e:
        logger.error(f"ParallelTaskExecutor test error: {e}")
        raise

result = framework.test_component("ParallelTaskExecutor", test_parallel_executor)
print(f"ParallelTaskExecutor Test: {result['status'].upper()}")

## 5️⃣ Test ResultSynthesizer Component

**Contract Requirements:**
- Input: Multiple heterogeneous results
- Output: Single synthesized result
- Type handling: Handle different result types
- Quality: Preserve important information

In [ ]:
def test_result_synthesizer():
    """Test ResultSynthesizer component"""
    try:
        from agents.result_synthesizer import ResultSynthesizer
        
        # Test 1: Synthesizer initialization
        synthesizer = ResultSynthesizer()
        framework.assert_contract(
            synthesizer is not None,
            "ResultSynthesizer must be instantiable"
        )
        
        # Test 2: Has synthesize method
        framework.assert_contract(
            hasattr(synthesizer, 'synthesize') and callable(synthesizer.synthesize),
            "ResultSynthesizer must have synthesize() method"
        )
        
        # Test 3: Synthesize heterogeneous results
        results = [
            {"agent": "analyzer", "findings": ["finding1", "finding2"]},
            {"agent": "generator", "content": "Generated content"},
            {"agent": "validator", "is_valid": True, "score": 0.95}
        ]
        
        synthesized = synthesizer.synthesize(results)
        framework.assert_contract(
            synthesized is not None,
            "Synthesize must return a result"
        )
        
        logger.info(f"   Synthesized {len(results)} results into: {type(synthesized).__name__}")
        TEST_RESULTS["components_tested"].append("ResultSynthesizer")
        
    except ImportError as e:
        logger.warning(f"⚠️  ResultSynthesizer import skipped: {e}")
    except Exception as e:
        logger.error(f"ResultSynthesizer test error: {e}")
        raise

result = framework.test_component("ResultSynthesizer", test_result_synthesizer)
print(f"ResultSynthesizer Test: {result['status'].upper()}")

## 6️⃣ Test OutputValidator Component

**Contract Requirements:**
- Input: Output from any agent
- Output: Validation result (bool or dict with details)
- Schema check: Validate against expected schema
- Content validation: Check for required fields

In [ ]:
def test_output_validator():
    """Test OutputValidator component"""
    try:
        from agents.output_validator import OutputValidator
        
        # Test 1: Validator initialization
        validator = OutputValidator()
        framework.assert_contract(
            validator is not None,
            "OutputValidator must be instantiable"
        )
        
        # Test 2: Has validate method
        framework.assert_contract(
            hasattr(validator, 'validate') and callable(validator.validate),
            "OutputValidator must have validate() method"
        )
        
        # Test 3: Validate good output
        good_output = {
            "status": "success",
            "result": "Valid output",
            "metadata": {"timestamp": datetime.now().isoformat()}
        }
        is_valid = validator.validate(good_output)
        framework.assert_contract(
            is_valid or isinstance(is_valid, dict),
            "Validate must return bool or dict"
        )
        
        # Test 4: Validate bad output
        bad_output = None
        is_bad_valid = validator.validate(bad_output)
        framework.assert_contract(
            not is_bad_valid or (isinstance(is_bad_valid, dict) and not is_bad_valid.get("valid", True)),
            "Validator must reject invalid output"
        )
        
        logger.info(f"   Good output valid: {is_valid}, Bad output valid: {is_bad_valid}")
        TEST_RESULTS["components_tested"].append("OutputValidator")
        
    except ImportError as e:
        logger.warning(f"⚠️  OutputValidator import skipped: {e}")
    except Exception as e:
        logger.error(f"OutputValidator test error: {e}")
        raise

result = framework.test_component("OutputValidator", test_output_validator)
print(f"OutputValidator Test: {result['status'].upper()}")

## 7️⃣ Test MemoryManager Component

**Contract Requirements:**
- Input: Data to store (any type)
- Output: Retrieval operations work correctly
- Persistence: Data survives across calls
- Cleanup: Memory can be cleared selectively

In [ ]:
def test_memory_manager():
    """Test MemoryManager component"""
    try:
        from agents.memory_manager import MemoryManager
        
        # Test 1: Memory manager initialization
        memory = MemoryManager()
        framework.assert_contract(
            memory is not None,
            "MemoryManager must be instantiable"
        )
        
        # Test 2: Has store and retrieve methods
        framework.assert_contract(
            hasattr(memory, 'store') and callable(memory.store),
            "MemoryManager must have store() method"
        )
        framework.assert_contract(
            hasattr(memory, 'retrieve') and callable(memory.retrieve),
            "MemoryManager must have retrieve() method"
        )
        
        # Test 3: Store and retrieve data
        test_key = "test_data"
        test_value = {"content": "test", "timestamp": datetime.now().isoformat()}
        
        memory.store(test_key, test_value)
        retrieved = memory.retrieve(test_key)
        
        framework.assert_contract(
            retrieved is not None,
            "Retrieved data must not be None"
        )
        
        # Test 4: Clear memory
        if hasattr(memory, 'clear'):
            memory.clear(test_key)
            retrieved_after_clear = memory.retrieve(test_key)
            logger.info(f"   Memory cleared, data after clear: {retrieved_after_clear}")
        
        logger.info(f"   Stored and retrieved: {test_key}")
        TEST_RESULTS["components_tested"].append("MemoryManager")
        
    except ImportError as e:
        logger.warning(f"⚠️  MemoryManager import skipped: {e}")
    except Exception as e:
        logger.error(f"MemoryManager test error: {e}")
        raise

result = framework.test_component("MemoryManager", test_memory_manager)
print(f"MemoryManager Test: {result['status'].upper()}")

## 8️⃣ Test Summary & Contract Compliance Report

**Summary of Component Tests:**

In [ ]:
summary = framework.get_summary()

print("=" * 60)
print("🧠 AGENT COMPONENTS TEST SUMMARY")
print("=" * 60)
print(f"\n📊 Metrics:")
print(f"   Total Tests: {summary['total_tests']}")
print(f"   Passed: {summary['passed']} ✅")
print(f"   Failed: {summary['failed']} ❌")
print(f"   Success Rate: {summary['success_rate']}")
print(f"   Duration: {summary['duration']:.2f}s")

print(f"\n🔧 Components Tested:")
for comp in sorted(summary['components_tested']):
    print(f"   ✓ {comp}")

if TEST_RESULTS["errors"]:
    print(f"\n⚠️  Errors Encountered:")
    for error in TEST_RESULTS["errors"]:
        print(f"   • {error['component']}: {error['error']}")

print("\n" + "=" * 60)
print("📋 Design Truth Contract Assessment")
print("=" * 60)

# Contract compliance assessment
contract_assessment = {
    "AgentRouter": {
        "required": ["route method", "task type handling", "agent instantiation"],
        "status": "TESTED" if "AgentRouter" in summary['components_tested'] else "NOT TESTED"
    },
    "TaskDecomposer": {
        "required": ["decompose method", "subtask generation", "dependency tracking"],
        "status": "TESTED" if "TaskDecomposer" in summary['components_tested'] else "NOT TESTED"
    },
    "ParallelTaskExecutor": {
        "required": ["execute method", "concurrent execution", "result ordering"],
        "status": "TESTED" if "ParallelTaskExecutor" in summary['components_tested'] else "NOT TESTED"
    },
    "ResultSynthesizer": {
        "required": ["synthesize method", "heterogeneous result handling"],
        "status": "TESTED" if "ResultSynthesizer" in summary['components_tested'] else "NOT TESTED"
    },
    "OutputValidator": {
        "required": ["validate method", "schema checking", "content validation"],
        "status": "TESTED" if "OutputValidator" in summary['components_tested'] else "NOT TESTED"
    },
    "MemoryManager": {
        "required": ["store method", "retrieve method", "persistence"],
        "status": "TESTED" if "MemoryManager" in summary['components_tested'] else "NOT TESTED"
    }
}

for component, assessment in contract_assessment.items():
    status_icon = "✅" if assessment['status'] == "TESTED" else "⏳"
    print(f"\n{status_icon} {component}")
    print(f"   Status: {assessment['status']}")
    for req in assessment['required']:
        print(f"   • {req}")

print("\n" + "=" * 60)